# Complete DeepSeekMoE Transformer

This notebook combines MLA attention, a dense early FFN, later MoE FFNs, and post-step expert-bias updates.

In [1]:
import sys
from pathlib import Path

import torch
import torch.nn.functional as F

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "model").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the repository root")
sys.path.insert(0, str(project_root))
torch.manual_seed(19)

In [2]:
from src.model import DeepSeekConfig, DeepSeekMoE, SwiGLUFFN, TinyDeepSeek
from src.training import MoELoadBalancer

config = DeepSeekConfig(
    vocab_size=256, dim=64, n_layers=3, n_dense_layers=1,
    n_heads=4, q_lora_rank=16, kv_lora_rank=8,
    qk_nope_head_dim=8, qk_rope_head_dim=4, v_head_dim=8,
    inter_dim=128, moe_inter_dim=32, n_routed_experts=4,
    n_activated_experts=2, max_seq_len=32,
)
model = TinyDeepSeek(config)
print("Layer FFNs:", [type(layer.ffn).__name__ for layer in model.layers])
print("Parameters:", sum(p.numel() for p in model.parameters()))

Layer FFNs: ['SwiGLUFFN', 'DeepSeekMoE', 'DeepSeekMoE']
Parameters: 135176


In [3]:
tokens = torch.randint(0, config.vocab_size, (2, 10))
inputs = tokens[:, :-1]
targets = tokens[:, 1:]

with MoELoadBalancer(model, gamma=0.01) as balancer:
    logits = model(inputs)
    loss = F.cross_entropy(
        logits.float().reshape(-1, config.vocab_size),
        targets.reshape(-1),
    )
    loss.backward()
    load_ratios = balancer.update()

print("Logits:", logits.shape)
print("Loss:", loss.item())
print("MoE max-load ratios:", load_ratios)
assert logits.shape == (2, 9, config.vocab_size)

Logits: torch.Size([2, 9, 256])
Loss: 5.797946929931641
MoE max-load ratios: [1.5555555820465088, 1.4444444179534912]


In [4]:
for layer in model.layers:
    if isinstance(layer.ffn, DeepSeekMoE):
        print("Router bias:", layer.ffn.router.expert_bias.tolist())
        print("Router gradient:", layer.ffn.router.gate.weight.grad.norm().item())

Router bias: [0.009999999776482582, -0.009999999776482582, 0.0, 0.009999999776482582]
Router gradient: 0.020919257774949074
Router bias: [0.009999999776482582, 0.009999999776482582, 0.0, -0.009999999776482582]
Router gradient: 0.026616694405674934
